## Six-class schema notice

This notebook uses the canonical 19-band, six-class pipeline shared with the backend
(`backend.config`) and the evaluation runner. Classifier parameters come from
`make_classifier`, so a notebook cannot silently drift from the model the dashboard serves.

Outputs are cleared. Everything embedded here previously was a four-class run on a
ten-band composite and does not describe this pipeline.

In [ ]:
import os
import sys

import ee
import geemap
from dotenv import load_dotenv
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../../..'))

load_dotenv()
ee.Initialize(project=os.getenv('EE_PROJECT_ID'))

os.makedirs('../../../doc/assets', exist_ok=True)

In [ ]:
# Canonical six-class / 19-band schema shared with backend and evaluation runner.
from backend.config import BANDS, FEATURE_COLLECTIONS, LAND_COVER_CLASSES, SEASONS
from backend.gee_classifier import (
    build_sentinel_composite,
    make_classifier,
    merge_feature_collections,
    sample_training_points,
)

bands = BANDS
season = SEASONS['summer']
class_names = [LAND_COVER_CLASSES[i]['name'] for i in LAND_COVER_CLASSES]
class_palette = [LAND_COVER_CLASSES[i]['color'].lstrip('#') for i in LAND_COVER_CLASSES]
legend_dict = {f"{info['name']} ({label})": info['color'].lstrip('#')
               for label, info in LAND_COVER_CLASSES.items()}

print('bands:', len(bands))
print('classes:', class_names)
print('season:', season)

In [ ]:
# ----- Madhya Pradesh boundary -----
madhya_pradesh = (
    ee.FeatureCollection('FAO/GAUL/2015/level1')
      .filter(ee.Filter.eq('ADM1_NAME', 'Madhya Pradesh'))
      .geometry()
)

# ----- Canonical feature composite -----
# build_sentinel_composite applies the same cloud mask, spectral indices, GLCM
# texture and Sentinel-1 SAR stack the backend classifies with. Rebuilding that
# by hand here is how the notebook and the served model drift apart.
composite = build_sentinel_composite(madhya_pradesh, season['start'], season['end'])
print('composite bands:', composite.bandNames().size().getInfo())

In [ ]:
# ----- Training points: every configured six-class asset -----
# Each asset carries its own `label`, so class identity travels with the points and
# nothing here has to re-assert it. That matters: an asset that lost its label
# property merges without complaint and then trains a silently empty class.
class_collections = {name: ee.FeatureCollection(path)
                     for name, path in FEATURE_COLLECTIONS.items()}

print('--- Raw points per asset ---')
total_points = 0
for name, collection in class_collections.items():
    histogram = collection.aggregate_histogram('label').getInfo()
    count = collection.size().getInfo()
    total_points += count
    print(f'{name}: {count} labels={histogram}')
    if not histogram:
        raise ValueError(f'{name} has no label property; it would train nothing')
print(f'Total points: {total_points}')

all_points = merge_feature_collections(FEATURE_COLLECTIONS.values())

# ----- Spatial block split (not a random per-point split) -----
# Labelled points come in clusters, so neighbouring 10 m pixels off the same lake
# or field land on both sides of a random split and the model gets scored against
# near-duplicates of its own training rows. Binning into ~11 km cells and sending
# whole cells to one side keeps a test point away from its training neighbour.
def assign_block(feature):
    coords = feature.geometry().coordinates()
    bx = ee.Number(coords.get(0)).multiply(10).floor()
    by = ee.Number(coords.get(1)).multiply(10).floor()
    return feature.set('blk', bx.multiply(31).add(by.multiply(17)).mod(10))

blocked_points = all_points.map(assign_block)

train_set = sample_training_points(
    blocked_points.filter(ee.Filter.lt('blk', 7)),
    start_date=season['start'], end_date=season['end'],
)
test_set = sample_training_points(
    blocked_points.filter(ee.Filter.gte('blk', 7)),
    start_date=season['start'], end_date=season['end'],
)

print('Train set size:', train_set.size().getInfo())
print('Test set size:', test_set.size().getInfo())

print('--- Per-class counts in train / test ---')
for index, name in enumerate(class_names):
    train_count = train_set.filter(ee.Filter.eq('label', index)).size().getInfo()
    test_count = test_set.filter(ee.Filter.eq('label', index)).size().getInfo()
    flag = '  <-- MISSING FROM TEST SET' if test_count == 0 else ''
    print(f'{name} (label={index}): train={train_count}, test={test_count}{flag}')

In [ ]:
# ----- Train Random Forest with the canonical backend parameters -----
classifier = make_classifier('rf').train(
    features=train_set,
    classProperty='label',
    inputProperties=bands,
)

test_classified = test_set.classify(classifier)
confusion_matrix = test_classified.errorMatrix('label', 'classification',
                                               list(LAND_COVER_CLASSES))

print('Overall Accuracy:', confusion_matrix.accuracy().getInfo())
print('Kappa Coefficient:', confusion_matrix.kappa().getInfo())

# Passing the full label list above fixes the matrix at 6x6, so a class with no
# test rows shows as an empty row instead of shifting every other class one
# column left and quietly reporting someone else's score as its own.
cm_array = confusion_matrix.array().getInfo()
print('\n--- Confusion matrix shape ---')
print('Rows:', len(cm_array), 'Cols:', len(cm_array[0]) if cm_array else 0)
assert len(cm_array) == len(LAND_COVER_CLASSES), 'matrix is not six-class'
print(cm_array)

# errorMatrix.order() gives the label order GEE actually used. Mapping accuracy
# through it rather than assuming positions 0..5 line up means a genuinely
# missing class reads as NaN, not a silent 0.0000.
print('\n--- Class-wise accuracy ---')
order = confusion_matrix.order().getInfo()
ca_by_label = dict(zip(order, [v for row in confusion_matrix.consumersAccuracy().getInfo() for v in row]))
pa_by_label = dict(zip(order, [v for row in confusion_matrix.producersAccuracy().getInfo() for v in row]))

for index, name in enumerate(class_names):
    ca = ca_by_label.get(index, float('nan'))
    pa = pa_by_label.get(index, float('nan'))
    print(f"{name}: Consumer's = {ca:.4f}, Producer's = {pa:.4f}")

plt.figure(figsize=(7, 6))
sns.heatmap(cm_array, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Six-Class Confusion Matrix - Madhya Pradesh')
plt.savefig('../../../doc/assets/multi_rf_mp_cm.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ----- Classify and visualise -----
classified = composite.select(bands).classify(classifier)
classified_smooth = classified.focalMode(radius=1, kernelType='square', units='pixels')

Map = geemap.Map(center=[23.5, 78.5], zoom=6)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'RGB Composite')
Map.addLayer(
    classified_smooth,
    {'min': 0, 'max': len(LAND_COVER_CLASSES) - 1, 'palette': class_palette},
    'RF Classification',
)
Map.add_legend(title='Land Cover', legend_dict=legend_dict)
Map.addLayer(madhya_pradesh, {}, 'Madhya Pradesh Boundary')
Map

In [ ]:
# ----- Class-wise area & export -----
# Madhya Pradesh is large enough that an interactive reduceRegion exceeds GEE
# limits, so both the area table and the raster go out as batch tasks.
area_image = ee.Image.pixelArea().addBands(classified_smooth)
areas = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
    geometry=madhya_pradesh,
    scale=30,
    maxPixels=1e13,
    tileScale=16,
)

task_area = ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection([ee.Feature(None, areas)]),
    description='RF_Multi_MP_Area_Stats',
    fileFormat='CSV',
)
task_area.start()
print('Area batch task started; check the Earth Engine Tasks tab.')

task_image = ee.batch.Export.image.toDrive(
    image=classified_smooth,
    description='RF_Multi_MP_Classification',
    folder='GEE_Exports',
    region=madhya_pradesh,
    scale=30,
    maxPixels=1e13,
)
task_image.start()
print('Classification image export batch task started.')